In [1]:
import numpy as np
import cvxpy as cp
import mosek
import random
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [16]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def ranktoset (A):
    A = list(A)
    sets = [[A[0]]]
    for i in range(1,len(A)):
        new = A[0:i+1]
        sets.append(new)
    return(sets)

def makesetflex (A, B):    # we assume A is non-empty
    N = len(A)
    B = list(B)
    M = len(B)
    added = []
    for i in range(M):
        new = B[0:i+1]
        for k in range(N):
            if len(A[k])==len(new) and len(np.intersect1d(A[k],new))==len(new):
                break
            if k == N-1:
                A.append(new)
                added.append(new)
    return(A,added)


def countsets(sets):
    m = len(sets)
    count = 0
    for k in range(m):
        count = count + len(sets[k])
    return(count)

def convertlist(sets):
    Output = []
    for temp in sets:
        for elem in temp:
            Output.append(elem)
    return(Output)

def solvenominal (sets,p,R,r,m,r_f,c):
    N = len(p)
    I = len(R[0])
    M = len(sets)
    v = cp.Variable((M,N))
    lbda = cp.Variable(M, nonneg = True)
    a = cp.Variable(I)
    alpha = cp.Variable(1)
    beta = cp.Variable(1)
    gamma = cp.Variable(1,nonneg = True)
    t = cp.Variable(N, nonneg = True)
    z2 = 0
    z4 = 0
    constraints = []
    for i in range(N):
        lbdasum = 0
        for j in range(M):
            if i in sets[j]:
                constraints.append(v[j][i] >= 0)
                lbdasum = lbdasum + lbda[j]
            else:
                constraints.append(v[j][i] >= 0)
        constraints.append((-R @ a)[i] - lbdasum - beta <= 0)
        z4 = z4 + p[i]*t[i]
        constraints.append(-alpha + cp.sum(v[0:M:1,i]) + cp.kl_div(gamma, t[i]) + gamma - t[i] <= 0)
    for j in range(M):
        z1 = -cp.min(v[j,sets[j]])*(1-m)+lbda[j]
        z2 = z2 + cp.pos(z1)
    constraints.append(a<= 1)
    constraints.append(a>=0)
    constraints.append(cp.sum(a)<=1)
    constraints.append(alpha + beta + gamma * (r-1) - (1-cp.sum(a))*r_f + z4 + z2 <= c)
    
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)
    
    
def robustcheck(a,R,r,c,p,m,r_f):
    N = len(p)
    x = -R.dot(a)
    rank = np.argsort(R.dot(a))
    q_b = cp.Variable(N, nonneg = True)
    q = cp.Variable(N, nonneg=True)
    constraints = [cp.sum(q) == 1, cp.sum(q_b)==1]
    phi_cons = 0
    for i in range(N):
        z1 = q_b[rank[0:i+1]]
        z2 = q[rank[0:i+1]]
        v = -cp.neg(cp.sum(z2)/(1-m)-1)+1
        constraints.append(cp.sum(z1)-v <= 0)
        phi_cons = phi_cons -cp.entr(q[i]) - q[i]*np.log(p[i])
    constraints.append(phi_cons <= r)
    obj = cp.Maximize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    print(prob.value - (1-np.sum(a))*r_f)
    print(c)
    return(prob.value - (1-np.sum(a))*r_f <= c+1e-5)

In [38]:
def cutting_plane(R,r,c,p,m,r_f,sets):
    nonstop = True
    iterations = 1
    N = len(sets)
    oldrank = list(np.arange(N))
    counter = 0
    while nonstop == True:
        [a,obj] = solvenominal(sets,p,R,r,m,r_f,c)
        newrank = np.argsort(R.dot(a))
        if np.array_equal(newrank,oldrank):
            if counter == 5:
                return(a,obj,iterations)
            print('ja',a,obj)
            np.random.shuffle(newrank)
            counter = counter + 1
            [sets,added] = makesetflex(sets, newrank)
            while added == []:
                np.random.shuffle(newrank)
                [sets,added] = makesetflex(sets, newrank)
        else:
            if counter >= 5:
                print('jammer', a, obj)
            oldrank = newrank
            iterations = iterations + 1
    

In [4]:
def riskcalc(a,R,p,alfa,r_f):
    x = R.dot(a)
    rank = np.argsort(R.dot(a))
    extra = 0
    if np.max(x) > 0:
        extra = np.max(x)
        x = x - np.max(x)
    x = -x
    N = len(p)
    risk = h_3(p[rank[0]],alfa)*x[rank[0]]
    for i in range(2,N+1):
        z1 = sum(p[rank[0:i]])
        z2 = sum(p[rank[0:i-1]])
        risk = risk + (h_3(z1,alfa)-h_3(z2,alfa))*x[rank[i-1]]
    risk = risk - extra
    print("the nominal risk of a:", risk)

    
def norisksolve(p,R,r_f):
    N = len(p)
    I = len(R[0])
    a = cp.Variable(I)
    constraints = [-1<=a, a<=1]
    obj = cp.Maximize((R@a).T @ p + (1-cp.sum(a))*r_f)
    prob = cp.Problem(obj,constraints)
    prob.solve(solver=cp.MOSEK)
    return(a.value, prob.value)

In [66]:
np.random.seed(5)

In [68]:
N=10
p = np.zeros(N)+1/N
I = 2
R = np.random.normal(0.05,0.2,size=(N,I))
print(R.transpose().dot(p))
#print(R)

[0.05103441 0.07753556]


[0.04585824 0.0462814 ]


In [65]:
r = 0.3
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 0.28
sets = [list(np.arange(N))]
cut_result = cutting_plane(R,r,c,p,m,r_f,sets)
print(cut_result)

ja [0.13319827 0.65822385] 0.03678033893810126
ja [0.13319828 0.65822389] 0.03678034105814025
ja [0.13319828 0.6582239 ] 0.0367803416061375
ja [0.13319828 0.6582239 ] 0.03678034165936711
ja [0.13319828 0.65822389] 0.03678034118021379
(array([0.13319828, 0.6582239 ]), 0.036780341662766634, 3)


In [63]:
r = 0.3
m = 0.9    # this is the parameter of the h function min(1, p/(1-m))
r_f = 0.001
c = 0.28
sets = ranktoset(np.arange(N))
cut_result = cutting_plane(R,r,c,p,m,r_f,sets)
print(cut_result)

ja [0.13360512 0.66023438] 0.03774793364465449
ja [0.13360513 0.66023442] 0.037747936157765026
ja [0.1336051  0.66023414] 0.03774792147812449
ja [0.13360513 0.6602344 ] 0.037747935300487145
ja [0.13360513 0.66023442] 0.03774793622776783
(array([0.13360513, 0.66023442]), 0.03774793622723942, 2)


In [87]:
riskcalc(cut_result[0],R,p,m,r_f)   

the nominal risk of a: 20.17807725697763


In [69]:
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets))]
resultall = solvenominal (psets,p,R,r,m,r_f,c)
print(resultall)

(array([1.7376405e-12, 1.0000000e+00]), 0.07753556302193333)


In [108]:
norisksolve(p,R,r_f)

(array([1.]), 2.772676007979409)

In [109]:
a = norisksolve(p,R,r_f)[0]
riskcalc(a,R,p,m,r_f)

the nominal risk of a: 37.317722909679404


In [380]:
def phi_div(p,q,r):
    phi_cons = 0
    for i in range(len(p)):
        phi_cons = q[i]*np.log(q[i]/p[i])+phi_cons
    print(phi_cons <= r)
    print(phi_cons)

In [385]:
q = np.zeros(N)+0.0001
q[0]=1-sum(q[1:len(q)])
phi_div(p,q,10)

True
3.371591603654162


In [111]:
sets

[[0],
 [8],
 [8, 4],
 [8, 4, 7],
 [8, 4, 7, 0],
 [8, 4, 7, 0, 3],
 [8, 4, 7, 0, 3, 5],
 [8, 4, 7, 0, 3, 5, 1],
 [8, 4, 7, 0, 3, 5, 1, 9],
 [8, 4, 7, 0, 3, 5, 1, 9, 6],
 [8, 4, 7, 0, 3, 5, 1, 9, 6, 2]]

In [110]:
min(R)

array([-37.31772291])

In [132]:
def seq(n):
    x = []
    for i in range(n):
        z = np.random.randint(n-i)
    return(x)

In [57]:
R[:,0]

array([ 0.10447669,  0.38019164, -0.31908983, -0.26907037,  0.2092692 ,
        0.10274756, -0.19596778, -0.08513067, -0.04615805, -0.35379261,
        0.13449206,  0.06585755,  0.19878489,  0.27257957,  0.52842166,
        0.08196316,  0.04900411, -0.1955083 ,  0.2475022 ,  0.02979745,
       -0.05403769,  0.01200758, -0.32810708,  0.24901488,  0.13208692,
       -0.17819742, -0.26717878,  0.24408791, -0.07477176,  0.22371511,
        0.0904044 , -0.01628549,  0.08509446, -0.02093044, -0.07686365,
       -0.19390561,  0.06993715,  0.18752482, -0.10788752,  0.11353385])